<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.2/2-memory-agent-advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Sistemas de Memoria Avanzados en LangChain

## Objetivos de Aprendizaje
- Comprender las limitaciones de la gestión manual de la memoria.
- Aprender a usar las clases de memoria de LangChain para automatizar el historial de chat.
- Implementar `ConversationBufferMemory` para un historial completo.
- Implementar `ConversationBufferWindowMemory` para limitar el tamaño del historial.
- Implementar `ConversationSummaryMemory` para resumir conversaciones y ahorrar tokens.
- Entender el puente hacia la planificación de agentes.

## El Problema con la Memoria Manual

En el notebook anterior, le dimos memoria a nuestro agente gestionando manualmente una lista `chat_history`. Aunque funcional, este enfoque tiene dos grandes inconvenientes:

1.  **Gestión Tediosa**: Actualizar la lista manualmente después de cada interacción es propenso a errores y no es escalable.
2.  **Límite de Contexto**: Enviar el historial completo en cada llamada consume tokens rápidamente. En conversaciones largas, esto puede exceder el límite de contexto del modelo, provocando errores o un alto costo.

LangChain resuelve esto con un sistema de **clases de Memoria** que automatizan el proceso y ofrecen estrategias para gestionar historiales largos.

### 1. Instalación y Configuración

In [ ]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai python-dotenv wikipedia


In [ ]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Agente y Herramientas (Sin Cambios)

Reutilizamos el mismo agente y herramientas. La innovación estará en cómo gestionamos la memoria.

In [ ]:
# LangChain v1: langchain.agents.create_agent (o langgraph)
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Este prompt antes se descargaba con `hub.pull("hwchase17/openai-tools-agent")`.
# LangChain ahora bloquea por defecto la descarga de prompts públicos, porque
# pueden contener objetos serializados no confiables. Definirlo aquí evita esa
# dependencia de red y, sobre todo, deja a la vista qué le estamos diciendo al
# agente: instrucción de sistema, historial, entrada del usuario y el
# "scratchpad" donde el agente va anotando sus llamadas a herramientas.
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Automatizando con `ConversationBufferMemory`

Esta es la memoria más básica. Simplemente almacena todos los mensajes en una variable (el "buffer") y los pasa en cada llamada. Replica lo que hicimos manualmente, pero de forma automática.

In [ ]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationBufferMemory

# memory_key debe coincidir con la variable de entrada en el prompt ('chat_history')
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

def chat_with_agent_buffer(query: str):
    # Carga el historial de la memoria
    history = memory.load_memory_variables({})["chat_history"]

    # Invoca al agente con el historial
    response = agent_executor.invoke({"input": query, "chat_history": history})

    # Guarda el nuevo par de pregunta/respuesta en la memoria
    memory.save_context({"input": query}, {"output": response["output"]})

    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_55501/170025402.py:6: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [ ]:
print("Primera pregunta...")
response1 = chat_with_agent_buffer("Háblame del planeta Marte")
print(f"Respuesta 1: {response1}")

print("Segunda pregunta de seguimiento...")
response2 = chat_with_agent_buffer("¿Tiene alguna luna?")
print(f"Respuesta 2: {response2}")

Primera pregunta...


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Marte planeta'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Marte'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

...

Marte es el cuarto planeta en orden de distancia al Sol,esta en su distancia promedio a aproximadamente 225 millones de km del Sol  y el segundo más pequeño del sistema solar, después de Mercurio. Recibió su nombre en homenaje al homónimo dios de la guerra de la mitología romana (Ares en la mitología griega), y también es conocido como «el planeta rojo»​​ debido a la apariencia rojiza​ que le confiere el óxido de hierro predominante en su superficie.

Marte es el cuarto planeta en orden de distancia al Sol y el segundo más pequeño del sistema solar, después de Mercurio. Recibió su nombre en honor al dios romano de la guerra y es conocido como **"el planeta rojo"** debido al óxido de hierro que domina su superficie.

> Finished chain.
Respuesta 1: Marte es el cuarto planeta en orden de distancia al Sol y el segundo más pequeño del sistema solar, después de Mercurio. Recibió su nombre en honor al dios romano de la guerra y es conocido como **"el planeta rojo"** debido al óxido de hierro que domina su superficie.
Segunda pregunta de seguimiento...


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Lunas de Marte'}`




Marte posee dos satélites naturales: Fobos y Deimos.​ Los dos son irregulares y se cree que fueron capturados del cercano cinturón de asteroides (región del sistema solar comprendida aproximadamente entre las órbitas de Marte y Júpiter).
Cabe resaltar que, a diferencia de la Luna terrestre, estos satélites no disipan las penumbras de la noche marciana, de este hecho se desprende la expresión coloquial más oscuro que las lunas de Marte.​


== Conjeturas especulativas previas ==
El astrónomo Johannes Kepler señaló a principios del siglo XVII que Marte debía tener dos satélites, en función de un razonamiento subordinado a la "armonía numérica":​ la Tierra tiene una luna y Júpiter, en el momento de realizar Kepler su afirmación, se creía que tenía únicamente las cuatro descubiertas por Galileo Galilei en 1610, por lo que a Marte, que estaba entre los dos planetas, le tocaban, proporcionalmente, dos.

Sí, Marte tiene dos lunas: **Fobos** y **Deimos**. Ambas son irregulares en forma y se cree que fueron asteroides capturados del cinturón de asteroides cercano. A diferencia de la Luna de la Tierra, no producen eclipses totales, lo que lleva a la expresión coloquial *"más oscuro que las lunas de Marte"*.

> Finished chain.
Respuesta 2: Sí, Marte tiene dos lunas: **Fobos** y **Deimos**. Ambas son irregulares en forma y se cree que fueron asteroides capturados del cinturón de asteroides cercano. A diferencia de la Luna de la Tierra, no producen eclipses totales, lo que lleva a la expresión coloquial *"más oscuro que las lunas de Marte"*.


Como puedes ver, la función `chat_with_agent_buffer` ahora se encarga de cargar y guardar el historial, haciendo el proceso mucho más limpio.

### 4. Gestionando el Contexto con `ConversationBufferWindowMemory`

Para evitar que el historial crezca sin control, podemos usar `ConversationBufferWindowMemory`. Esta memoria solo conserva un número `k` de las últimas interacciones.

Esto es un buen equilibrio entre tener contexto y no sobrecargar al LLM.

In [ ]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationBufferWindowMemory

# Creamos una nueva memoria, esta vez con k=1 (solo recuerda el último par de mensajes)
memory_window = ConversationBufferWindowMemory(k=1, memory_key="chat_history", return_messages=True)

def chat_with_agent_window(query: str):
    history = memory_window.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_window.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_55501/3895951803.py:6: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory_window = ConversationBufferWindowMemory(k=1, memory_key="chat_history", return_messages=True)


In [ ]:
print("Pregunta 1: ¿Quién fue Albert Einstein?")
chat_with_agent_window("¿Quién fue Albert Einstein?")

print("Pregunta 2: ¿Y Niels Bohr?")
chat_with_agent_window("¿Y Niels Bohr?")

print("Pregunta 3: ¿De qué trataba la primera pregunta que te hice?")
response3 = chat_with_agent_window("¿De qué trataba la primera pregunta que te hice?")
print(f"Respuesta 3: {response3}")

Pregunta 1: ¿Quién fue Albert Einstein?


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`




Albert Einstein pronunciación en alemán: /ˈalbɐt ˈaɪnʃtaɪn/ ();​ (Ulm, 14 de marzo de 1879-Princeton, Nueva Jersey, 18 de abril de 1955) fue un físico alemán de origen judío, nacionalizado después suizo, austriaco y estadounidense. Se le considera el científico más importante, conocido y popular del siglo XX.​​
En 1905, cuando era un joven físico desconocido, empleado en la Oficina de Patentes de Berna, publicó su teoría de la relatividad especial.

Albert Einstein (1879–1955) fue un físico alemán de origen judío, nacionalizado suizo, austriaco y estadounidense, considerado el científico más importante y popular del siglo XX. Revolucionó la física con su teoría de la relatividad especial en 1905, mientras trabajaba en la Oficina de Patentes de Berna.

> Finished chain.
Pregunta 2: ¿Y Niels Bohr?


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Niels Bohr'}`




...

Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Niels Bohr científico'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Niels Bohr (1885–1962) fue un físico danés que realizó contribuciones fundamentales a la comprensión de la estructura del átomo y la mecánica cuántica. Ganó el Premio Nobel de Física en 1922 por su trabajo sobre la estructura atómica y la radiación.

> Finished chain.
Pregunta 3: ¿De qué trataba la primera pregunta que te hice?


> Entering new AgentExecutor chain...


La primera pregunta que me hiciste fue: **"¿Y Niels Bohr?"**, lo que interpreté como un interés en obtener información sobre este físico danés. Por eso, te proporcioné un resumen sobre su vida y sus contribuciones científicas.

> Finished chain.
Respuesta 3: La primera pregunta que me hiciste fue: **"¿Y Niels Bohr?"**, lo que interpreté como un interés en obtener información sobre este físico danés. Por eso, te proporcioné un resumen sobre su vida y sus contribuciones científicas.


El agente probablemente no podrá responder a la tercera pregunta, porque con `k=1`, la interacción sobre Einstein ya fue descartada de la memoria para dar paso a la de Niels Bohr.

### 5. Ahorrando Tokens con `ConversationSummaryMemory`

Esta es la estrategia más avanzada. En lugar de guardar los mensajes completos, utiliza un LLM para crear un resumen de la conversación a medida que avanza. En cada nueva interacción, el resumen se actualiza y se pasa como contexto.

Es ideal para conversaciones muy largas donde los detalles específicos se vuelven menos importantes que el contexto general.

In [ ]:
# LangChain v1: persistencia de LangGraph (RunnableWithMessageHistory, el paso
# intermedio que usa este notebook, también quedó deprecado)
from langchain_classic.memory import ConversationSummaryMemory

# Esta memoria necesita un LLM para hacer los resúmenes
memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

def chat_with_agent_summary(query: str):
    history = memory_summary.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_summary.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/hc/s32qf6c501b8c_vjpqkvxdpw0000gn/T/ipykernel_55501/193633793.py:6: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [ ]:
print("Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")
chat_with_agent_summary("Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")

print("Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?")
chat_with_agent_summary("¿Qué ciudades me recomiendas visitar en un primer viaje?")

print("Pregunta 3: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_summary("¿Sobre qué estábamos hablando?")
print(f"Respuesta 3: {response3_summary}")

Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?


> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'mejor época para viajar a Japón'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'mejor época para visitar Japón'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

...

Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?


> Entering new AgentExecutor chain...


Para un primer viaje a Japón, te recomendaría visitar estas ciudades, cada una con experiencias únicas:

1. **Tokio**: La capital es una mezcla vibrante de tradición y modernidad. No te pierdas lugares como el Templo Senso-ji en Asakusa, el barrio de Shibuya con su famoso cruce, y el moderno distrito de Shibuya o Akihabara para los amantes de la tecnología y el anime. También puedes disfrutar de la gastronomía en el mercado de Tsukiji o en restaurantes de ramen por toda la ciudad.

2. **Kioto**: Es el corazón cultural de Japón. Aquí podrás visitar templos históricos como el Kinkaku-ji (Pabellón Dorado) o el Fushimi Inari con sus miles de torii rojos. Kioto es famosa por sus barrios tradicionales como Gion, donde podrías avistar geishas, y por sus jardines zen. Además, es ideal para experimentar ceremonias del té y talleres de artesanía.

3. **Osaka**: Conocida como la "cocina de Japón", es perfecta para los foodies. No dejes de probar el takoyaki (bolas de pulpo) y el okonomiyaki (tort

Pregunta 3: ¿Sobre qué estábamos hablando?


> Entering new AgentExecutor chain...


Estábamos hablando sobre la organización de un viaje a Japón. Te ayudé a determinar la mejor época para visitarlo según tus preferencias y te recomendé cinco ciudades clave para un primer viaje: **Tokio, Kioto, Osaka, Hiroshima y Nara**. Además, mencioné otras opciones como **Hakone** o **Kanazawa** si tienes tiempo extra.

¿Te gustaría profundizar en alguna ciudad, obtener más detalles sobre actividades específicas o necesitas ayuda con otro aspecto del viaje?

> Finished chain.


Respuesta 3: Estábamos hablando sobre la organización de un viaje a Japón. Te ayudé a determinar la mejor época para visitarlo según tus preferencias y te recomendé cinco ciudades clave para un primer viaje: **Tokio, Kioto, Osaka, Hiroshima y Nara**. Además, mencioné otras opciones como **Hakone** o **Kanazawa** si tienes tiempo extra.

¿Te gustaría profundizar en alguna ciudad, obtener más detalles sobre actividades específicas o necesitas ayuda con otro aspecto del viaje?


El agente debería poder responder correctamente, ya que la memoria no contiene los mensajes literales, sino un resumen de que el usuario está planeando un viaje a Japón.

## Conclusiones y Transición al Módulo IL2.3

Hemos visto cómo las clases de memoria de LangChain nos permiten crear agentes conversacionales mucho más robustos y eficientes, superando los desafíos de la gestión manual y el tamaño del contexto.

**Resumen de Estrategias de Memoria:**
- **`ConversationBufferMemory`**: Simple y completa. Ideal para conversaciones cortas donde cada detalle importa.
- **`ConversationBufferWindowMemory`**: Eficiente y balanceada. Perfecta para chatbots de servicio al cliente que necesitan contexto reciente.
- **`ConversationSummaryMemory`**: Ahorradora de tokens y escalable. La mejor opción para asistentes de largo plazo y análisis de conversaciones extensas.

Con un agente que puede recordar de manera efectiva, hemos completado una pieza fundamental del rompecabezas. Sin embargo, las conversaciones del mundo real a menudo implican tareas que no se resuelven con una sola herramienta o en un solo paso.

**¿Qué pasa si el agente necesita buscar información, luego calcular algo con esa información y finalmente enviar un correo electrónico?**

Esto requiere que el agente pueda **planificar** una secuencia de acciones. Debe ser capaz de descomponer un objetivo complejo en pasos más pequeños y ejecutarlos en el orden correcto.

Esta es la transición perfecta al **Módulo IL2.3: Planificación y Orquestación de Agentes**, donde exploraremos cómo los agentes pueden crear, seguir y adaptar planes para resolver problemas complejos y multi-paso.